In [2]:
# Add at start of notebook
!pip install transformers
!pip install accelerate

In [4]:
# Import required libraries
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import pandas as pd
import os
from tqdm import tqdm
import numpy as np

def load_class_list():
    """
    Load valid class names from a space-delimited text file.
    
    Returns:
        list: List of valid class names for classification
    """
    classes_df = pd.read_csv('/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/vlg-dataset/classes.txt', 
                            delimiter='\s+', header=None, names=['index', 'class_name'])
    valid_classes = classes_df['class_name'].tolist()
    return valid_classes

def get_enhanced_prompts(classes):
    prompts = []
    for cls in classes:
        class_name = cls.replace('+', ' ')  # Clean class names with '+'
        # Multiple templates with different contexts for robustness
        templates = [
            f"a photograph of a {class_name} in its natural habitat",    # Weight: 1.2
            f"a high resolution photo of a {class_name}",                # Weight: 1.0
            f"a close-up wildlife photo of a {class_name}",             # Weight: 1.1
            f"a professional photograph of a {class_name}",             # Weight: 1.0
            f"a clear detailed shot of a {class_name}",                 # Weight: 1.1
            f"a nature photograph showing a {class_name}"               # Weight: 1.0
        ]
        prompts.extend(templates)
    return prompts, len(templates)

def apply_advanced_tta(image):
    augmented = []
    width, height = image.size
    
    # Original image (weight: 1.0)
    augmented.append(image)
    
    # Horizontal flip for orientation robustness (weight: 0.8)
    augmented.append(image.transpose(Image.FLIP_LEFT_RIGHT))
    
    # Center crops at different scales (weight: 0.9 each)
    for scale in [0.85, 0.95]:  # Two different crop scales for robustness
        crop_size = int(min(width, height) * scale)
        left = (width - crop_size) // 2
        top = (height - crop_size) // 2
        crop = image.crop((left, top, left + crop_size, top + crop_size))
        augmented.append(crop)
    
    return augmented

def predict_with_model(model, processor, image, prompts, templates_per_class, valid_classes):
    """
    Generate predictions using a single CLIP model with augmentation and prompt ensemble.
    
    Args:
        model: CLIP model instance
        processor: CLIP processor instance
        image: Input image
        prompts: List of text prompts
        templates_per_class: Number of templates per class
        valid_classes: List of valid class names
    
    Returns:
        numpy.ndarray: Class probability distribution
    """
    augmented_images = apply_advanced_tta(image)
    all_probs = []
    
    for aug_image in augmented_images:
        # Process image-text pairs through CLIP
        inputs = processor(
            images=aug_image,
            text=prompts,
            return_tensors="pt",
            padding=True
        ).to('cuda')
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)
            
            # Reshape probabilities to handle multiple templates
            probs = probs.cpu().numpy().reshape(-1, len(valid_classes), templates_per_class)
            
            # Apply template weights (determined empirically)
            template_weights = np.array([1.2, 1.0, 1.1, 1.0, 1.1, 1.0])
            weighted_probs = probs * template_weights.reshape(1, 1, -1)
            avg_probs = weighted_probs.mean(axis=2)
            all_probs.append(avg_probs)
    
    # Weighted average of augmentations
    aug_weights = np.array([1.0, 0.8, 0.9, 0.9])[:len(all_probs)]
    aug_weights = aug_weights / aug_weights.sum()
    final_probs = np.average(all_probs, axis=0, weights=aug_weights)
    
    return final_probs

def predict_classes():
    """
    Main prediction pipeline that processes all images and generates predictions.
    Handles model ensemble, augmentation, and error cases.
    """
    # Initialize CLIP models (base: 0.4 weight, large: 0.6 weight)
    models = {
        'base': (CLIPModel.from_pretrained("openai/clip-vit-base-patch32"), 
                CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")),
        'large': (CLIPModel.from_pretrained("openai/clip-vit-large-patch14"), 
                 CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14"))
    }
    
    # Move models to GPU
    for model, processor in models.values():
        model.to('cuda')
    
    valid_classes = load_class_list()
    prompts, templates_per_class = get_enhanced_prompts(valid_classes)
    predictions = []
    
    # Process each image with progress bar
    for image_name in tqdm(os.listdir(TEST_PATH)):
        if not image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
            
        try:
            image_path = os.path.join(TEST_PATH, image_name)
            image = Image.open(image_path).convert('RGB')
            
            # Get predictions from each model
            all_model_probs = []
            for model, processor in models.values():
                model_probs = predict_with_model(
                    model, processor, image, prompts, 
                    templates_per_class, valid_classes
                )
                all_model_probs.append(model_probs)
            
            # Weighted ensemble of models (large model given more weight)
            model_weights = np.array([0.4, 0.6])  
            final_probs = np.average(all_model_probs, axis=0, weights=model_weights)
            
            pred_idx = final_probs.argmax()
            predicted_class = valid_classes[pred_idx]
            
            predictions.append({
                'image_id': image_name,
                'class': predicted_class
            })
            
        except Exception as e:
            print(f"Error processing {image_name}: {str(e)}")
            # Fallback to 'antelope' class on error (common class in dataset)
            predictions.append({
                'image_id': image_name,
                'class': 'antelope'
            })
    
    # Save predictions and print summary
    predictions_df = pd.DataFrame(predictions)
    predictions_df.to_csv('predictions.csv', index=False)
    
    print("\nPrediction Summary:")
    print(f"Total images processed: {len(predictions)}")
    print("\nClass distribution:")
    print(predictions_df['class'].value_counts())

if __name__ == "__main__":
    # Configure paths and performance settings
    TEST_PATH = '/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/vlg-dataset/test'
    torch.backends.cudnn.benchmark = True  # Enable cuDNN auto-tuner
    
    predict_classes()

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
100%|██████████| 3000/3000 [51:42<00:00,  1.03s/it]


Prediction Summary:
Total images processed: 3000

Class distribution:
class
squirrel           121
collie             121
horse              121
rhinoceros         120
sheep              120
rabbit             119
fox                118
gorilla            118
moose              117
chimpanzee         115
humpback+whale      61
buffalo             54
spider+monkey       51
walrus              51
otter               50
hamster             49
mole                48
siamese+cat         47
wolf                47
deer                47
skunk               46
cow                 46
lion                45
dolphin             45
dalmatian           45
bobcat              45
grizzly+bear        45
leopard             45
giraffe             45
giant+panda         45
raccoon             45
tiger               45
zebra               45
elephant            45
bat                 45
pig                 45
german+shepherd     44
persian+cat         44
polar+bear          44
hippopotamus        44
chi